In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
# Nom du fichier Excel
file_path = r"C:\Users\AdMin\Desktop\zilo prepa commd dffff.xlsx"

# Lecture du fichier
df = pd.read_excel(file_path)

# Vérification
print("=== APERÇU INITIAL DE LA BASE ===")
print(df.head())

print("\n=== DIMENSIONS DE LA BASE ===")
print(df.shape)

print("\n=== NOMS DES COLONNES ===")
print(df.columns.tolist())

=== APERÇU INITIAL DE LA BASE ===
   N°de BP     Date de Création  Date de Préparation    Date de Contrôle   \
0  2500784  2025-01-15 00:00:00  2025-01-15 00:00:00  2025-01-15 00:00:00   
1  2500630  2025-01-13 00:00:00  2025-01-13 00:00:00  2025-01-14 00:00:00   
2  2500667  2025-01-13 00:00:00  2025-01-13 00:00:00  2025-01-15 00:00:00   
3  2500769  2025-01-15 00:00:00  2025-01-15 00:00:00  2025-01-15 00:00:00   
4  2500778  2025-01-15 00:00:00  2025-01-15 00:00:00  2025-01-15 00:00:00   

  Code Client  Nombre de ligne Heure de début de préparation  \
0     PH00312               23                      15:35:00   
1     GR00058               35                      10:35:00   
2     GR00068              110                      11:30:00   
3     PH05129               41                      13:42:00   
4     PH00086               52                      13:40:00   

  Heure de fin de préparation  Quantité des produits CD temps de preparation  
0                    16:15:00          

In [4]:
# Supprimer les espaces inutiles dans les noms de colonnes
df.columns = df.columns.str.strip()

print("\n=== COLONNES APRÈS NETTOYAGE ===")
print(df.columns.tolist())


=== COLONNES APRÈS NETTOYAGE ===
['N°de BP', 'Date de Création', 'Date de Préparation', 'Date de Contrôle', 'Code Client', 'Nombre de ligne', 'Heure de début de préparation', 'Heure de fin de préparation', 'Quantité des produits CD', 'temps de preparation']


In [5]:
# Garder uniquement les variables utiles
data = df[[
    "N°de BP",
    "Date de Création",
    "Date de Préparation",
    "Code Client",
    "Nombre de ligne",
    "Quantité des produits CD",
    "temps de preparation"
]].copy()

print("\n=== APERÇU DES VARIABLES UTILES ===")
print(data.head())

print("\n=== DIMENSIONS APRÈS SÉLECTION DES VARIABLES ===")
print(data.shape)


=== APERÇU DES VARIABLES UTILES ===
   N°de BP     Date de Création  Date de Préparation Code Client  \
0  2500784  2025-01-15 00:00:00  2025-01-15 00:00:00     PH00312   
1  2500630  2025-01-13 00:00:00  2025-01-13 00:00:00     GR00058   
2  2500667  2025-01-13 00:00:00  2025-01-13 00:00:00     GR00068   
3  2500769  2025-01-15 00:00:00  2025-01-15 00:00:00     PH05129   
4  2500778  2025-01-15 00:00:00  2025-01-15 00:00:00     PH00086   

   Nombre de ligne  Quantité des produits CD temps de preparation  
0               23                     500.0             00:40:00  
1               35                   29808.0             01:45:00  
2              110                   10921.0             04:50:00  
3               41                    1770.0             00:52:00  
4               52                     777.0             01:00:00  

=== DIMENSIONS APRÈS SÉLECTION DES VARIABLES ===
(2275, 7)


In [6]:
# Conversion de la colonne temps en durée
data["temps de preparation"] = pd.to_timedelta(
    data["temps de preparation"].astype(str),
    errors="coerce"
)

# Transformation en minutes
data["temps_prep_min"] = data["temps de preparation"].dt.total_seconds() / 60

print("\n=== TEMPS CONVERTI EN MINUTES ===")
print(data[["temps de preparation", "temps_prep_min"]].head(10))


=== TEMPS CONVERTI EN MINUTES ===
  temps de preparation  temps_prep_min
0      0 days 00:40:00            40.0
1      0 days 01:45:00           105.0
2      0 days 04:50:00           290.0
3      0 days 00:52:00            52.0
4      0 days 01:00:00            60.0
5      0 days 03:25:00           205.0
6      0 days 00:34:00            34.0
7      0 days 00:24:00            24.0
8      0 days 02:45:00           165.0
9      0 days 02:55:00           175.0


In [7]:
# Conversion en numérique
data["Nombre de ligne"] = pd.to_numeric(data["Nombre de ligne"], errors="coerce")
data["Quantité des produits CD"] = pd.to_numeric(data["Quantité des produits CD"], errors="coerce")

print("\n=== TYPES DES VARIABLES ===")
print(data.dtypes)


=== TYPES DES VARIABLES ===
N°de BP                               int64
Date de Création                     object
Date de Préparation                  object
Code Client                          object
Nombre de ligne                       int64
Quantité des produits CD            float64
temps de preparation        timedelta64[ns]
temps_prep_min                      float64
dtype: object


In [8]:
# Nombre de lignes avant suppression
print("\nNombre de lignes avant suppression des valeurs manquantes :", len(data))

# Suppression des lignes incomplètes
data = data.dropna()

# Nombre de lignes après suppression
print("Nombre de lignes après suppression des valeurs manquantes :", len(data))


Nombre de lignes avant suppression des valeurs manquantes : 2275
Nombre de lignes après suppression des valeurs manquantes : 2267


In [9]:
# Filtrage des valeurs positives
data = data[
    (data["Nombre de ligne"] > 0) &
    (data["Quantité des produits CD"] > 0) &
    (data["temps_prep_min"] > 0)
]

print("\n=== NOMBRE DE LIGNES APRÈS FILTRAGE DES VALEURS POSITIVES ===")
print(len(data))


=== NOMBRE DE LIGNES APRÈS FILTRAGE DES VALEURS POSITIVES ===
2267


In [10]:
# Calcul des bornes IQR
Q1 = data["temps_prep_min"].quantile(0.25)
Q3 = data["temps_prep_min"].quantile(0.75)
IQR = Q3 - Q1

borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

print("\n=== BORNES POUR DÉTECTER LES VALEURS ABERRANTES ===")
print("Borne inférieure :", borne_inf)
print("Borne supérieure :", borne_sup)

# Filtrage
data = data[
    (data["temps_prep_min"] >= borne_inf) &
    (data["temps_prep_min"] <= borne_sup)
]

print("\n=== NOMBRE DE LIGNES APRÈS SUPPRESSION DES VALEURS ABERRANTES ===")
print(len(data))


=== BORNES POUR DÉTECTER LES VALEURS ABERRANTES ===
Borne inférieure : -152.5
Borne supérieure : 347.5

=== NOMBRE DE LIGNES APRÈS SUPPRESSION DES VALEURS ABERRANTES ===
2203


In [11]:
print("\n=== STATISTIQUES DESCRIPTIVES ===")
print(data[["Nombre de ligne", "Quantité des produits CD", "temps_prep_min"]].describe())


=== STATISTIQUES DESCRIPTIVES ===
       Nombre de ligne  Quantité des produits CD  temps_prep_min
count      2203.000000               2203.000000     2203.000000
mean         57.877440               7437.219700      101.506128
std          37.139126              12342.587208       79.537647
min           1.000000                  8.000000        2.000000
25%          27.000000                814.000000       35.000000
50%          45.000000               3213.000000       77.000000
75%          87.000000               8576.500000      150.000000
max         188.000000             160055.000000      346.000000


In [12]:
# Variables explicatives
X = data[["Nombre de ligne", "Quantité des produits CD"]]

# Variable cible
y = data["temps_prep_min"]

print("\n=== APERÇU DE X ===")
print(X.head())

print("\n=== APERÇU DE y ===")
print(y.head())


=== APERÇU DE X ===
   Nombre de ligne  Quantité des produits CD
0               23                     500.0
1               35                   29808.0
2              110                   10921.0
3               41                    1770.0
4               52                     777.0

=== APERÇU DE y ===
0     40.0
1    105.0
2    290.0
3     52.0
4     60.0
Name: temps_prep_min, dtype: float64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\n=== TAILLE DES ÉCHANTILLONS ===")
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)


=== TAILLE DES ÉCHANTILLONS ===
X_train : (1762, 2)
X_test  : (441, 2)
y_train : (1762,)
y_test  : (441,)


In [14]:
# Création du modèle
model = LinearRegression()

# Entraînement du modèle
model.fit(X_train, y_train)

print("\n=== MODÈLE ENTRAÎNÉ AVEC SUCCÈS ===")


=== MODÈLE ENTRAÎNÉ AVEC SUCCÈS ===


In [15]:
# Prédiction sur les données de test
y_pred = model.predict(X_test)

# Tableau de comparaison
resultats_test = pd.DataFrame({
    "Temps réel (min)": y_test.values,
    "Temps prédit (min)": y_pred
})

print("\n=== EXEMPLE DE PRÉDICTIONS ===")
print(resultats_test.head(20))


=== EXEMPLE DE PRÉDICTIONS ===
    Temps réel (min)  Temps prédit (min)
0               35.0           55.946197
1              170.0          199.510258
2               45.0           51.636963
3               55.0           56.916692
4               35.0           33.458625
5               65.0           70.355195
6                9.0           57.480849
7               30.0           63.353088
8               85.0          152.371499
9               54.0           44.543401
10              20.0           29.119851
11              23.0           42.864770
12             340.0          202.152509
13              50.0           40.819085
14             155.0          158.334213
15              90.0           70.071336
16              45.0           48.545631
17              22.0           34.814998
18             100.0           86.850143
19              10.0           41.903053


In [16]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n=== PERFORMANCE DU MODÈLE ===")
print(f"MAE  = {mae:.2f} minutes")
print(f"RMSE = {rmse:.2f} minutes")
print(f"R²   = {r2:.4f}")


=== PERFORMANCE DU MODÈLE ===
MAE  = 30.78 minutes
RMSE = 45.84 minutes
R²   = 0.6742


In [17]:
intercept = model.intercept_
coef_lignes = model.coef_[0]
coef_quantite = model.coef_[1]

print("\n=== ÉQUATION DU MODÈLE ===")
print(f"Temps_prep_min = {intercept:.4f} + ({coef_lignes:.4f} × Nombre de ligne) + ({coef_quantite:.6f} × Quantité)")


=== ÉQUATION DU MODÈLE ===
Temps_prep_min = 4.0275 + (1.4546 × Nombre de ligne) + (0.001786 × Quantité)


In [18]:
# Prédiction sur toutes les lignes
data["delai_standard_prevu_min"] = model.predict(X)

# Calcul de l'écart
data["ecart_min"] = data["temps_prep_min"] - data["delai_standard_prevu_min"]

# Taux de réalisation
data["taux_realisation_%"] = (data["temps_prep_min"] / data["delai_standard_prevu_min"]) * 100

print("\n=== APERÇU DES RÉSULTATS FINAUX ===")
print(data[[
    "N°de BP",
    "Nombre de ligne",
    "Quantité des produits CD",
    "temps_prep_min",
    "delai_standard_prevu_min",
    "ecart_min",
    "taux_realisation_%"
]].head(10))


=== APERÇU DES RÉSULTATS FINAUX ===
   N°de BP  Nombre de ligne  Quantité des produits CD  temps_prep_min  \
0  2500784               23                     500.0            40.0   
1  2500630               35                   29808.0           105.0   
2  2500667              110                   10921.0           290.0   
3  2500769               41                    1770.0            52.0   
4  2500778               52                     777.0            60.0   
5  2500688              104                   11834.0           205.0   
6  2500768               37                     717.0            34.0   
7  2500729               24                     760.0            24.0   
8  2500640              118                   22143.0           165.0   
9  2500668               84                    1991.0           175.0   

   delai_standard_prevu_min   ecart_min  taux_realisation_%  
0                 38.375990    1.624010          104.231837  
1                108.168688   -3.16

In [22]:
def statut_performance(taux):
    if taux < 90:
        return "Plus rapide que le standard"
    elif taux <= 110:
        return "Conforme au standard"
    else:
        return "Plus lent que le standard"

data["statut_performance"] = data["taux_realisation_%"].apply(statut_performance)

print("\n=== RÉPARTITION DES STATUTS ===")
print(data["statut_performance"].value_counts())


=== RÉPARTITION DES STATUTS ===
statut_performance
Plus rapide que le standard    1180
Plus lent que le standard       620
Conforme au standard            403
Name: count, dtype: int64


In [23]:
# Classes du nombre de lignes
data["classe_lignes"] = pd.cut(
    data["Nombre de ligne"],
    bins=[0, 5, 10, 20, 50, 1000],
    labels=["1-5", "6-10", "11-20", "21-50", "51 et +"]
)

# Classes de quantité
data["classe_quantite"] = pd.qcut(
    data["Quantité des produits CD"],
    q=4,
    labels=["Faible", "Moyenne", "Élevée", "Très élevée"],
    duplicates="drop"
)

print("\n=== APERÇU DES CLASSES ===")
print(data[[
    "Nombre de ligne",
    "Quantité des produits CD",
    "classe_lignes",
    "classe_quantite"
]].head(10))


=== APERÇU DES CLASSES ===
   Nombre de ligne  Quantité des produits CD classe_lignes classe_quantite
0               23                     500.0         21-50          Faible
1               35                   29808.0         21-50     Très élevée
2              110                   10921.0       51 et +     Très élevée
3               41                    1770.0         21-50         Moyenne
4               52                     777.0       51 et +          Faible
5              104                   11834.0       51 et +     Très élevée
6               37                     717.0         21-50          Faible
7               24                     760.0         21-50          Faible
8              118                   22143.0       51 et +     Très élevée
9               84                    1991.0       51 et +         Moyenne


In [24]:
standards_classes = data.groupby(
    ["classe_lignes", "classe_quantite"],
    observed=True
).agg(
    nb_commandes=("temps_prep_min", "count"),
    temps_reel_moyen_min=("temps_prep_min", "mean"),
    temps_standard_moyen_min=("delai_standard_prevu_min", "mean"),
    temps_reel_median_min=("temps_prep_min", "median")
).reset_index()

print("\n=== STANDARDS PAR CLASSES ===")
print(standards_classes)


=== STANDARDS PAR CLASSES ===
   classe_lignes classe_quantite  nb_commandes  temps_reel_moyen_min  \
0            1-5          Faible             1              5.000000   
1           6-10          Faible             1             10.000000   
2          11-20          Faible           208             22.937500   
3          11-20         Moyenne            37             35.054054   
4          11-20          Élevée            17             40.705882   
5          11-20     Très élevée             8             80.250000   
6          21-50          Faible           340             34.576471   
7          21-50         Moyenne           290             54.103448   
8          21-50          Élevée           159             77.874214   
9          21-50     Très élevée           176            124.272727   
10       51 et +          Faible             3             49.666667   
11       51 et +         Moyenne           222            108.391892   
12       51 et +          Élevée 

In [25]:
performance_modele = pd.DataFrame({
    "Indicateur": ["MAE", "RMSE", "R2", "Intercept", "Coef_Nombre_de_ligne", "Coef_Quantite"],
    "Valeur": [mae, rmse, r2, intercept, coef_lignes, coef_quantite]
})

print("\n=== PERFORMANCE DU MODÈLE SOUS FORME DE TABLE ===")
print(performance_modele)


=== PERFORMANCE DU MODÈLE SOUS FORME DE TABLE ===
             Indicateur     Valeur
0                   MAE  30.784808
1                  RMSE  45.836944
2                    R2   0.674240
3             Intercept   4.027496
4  Coef_Nombre_de_ligne   1.454592
5         Coef_Quantite   0.001786


In [26]:
resume_bi = data[[
    "N°de BP",
    "Date de Création",
    "Date de Préparation",
    "Code Client",
    "Nombre de ligne",
    "Quantité des produits CD",
    "temps_prep_min",
    "delai_standard_prevu_min",
    "ecart_min",
    "taux_realisation_%",
    "statut_performance",
    "classe_lignes",
    "classe_quantite"
]].copy()

print("\n=== APERÇU DU FICHIER RÉSUMÉ POUR POWER BI ===")
print(resume_bi.head())


=== APERÇU DU FICHIER RÉSUMÉ POUR POWER BI ===
   N°de BP     Date de Création  Date de Préparation Code Client  \
0  2500784  2025-01-15 00:00:00  2025-01-15 00:00:00     PH00312   
1  2500630  2025-01-13 00:00:00  2025-01-13 00:00:00     GR00058   
2  2500667  2025-01-13 00:00:00  2025-01-13 00:00:00     GR00068   
3  2500769  2025-01-15 00:00:00  2025-01-15 00:00:00     PH05129   
4  2500778  2025-01-15 00:00:00  2025-01-15 00:00:00     PH00086   

   Nombre de ligne  Quantité des produits CD  temps_prep_min  \
0               23                     500.0            40.0   
1               35                   29808.0           105.0   
2              110                   10921.0           290.0   
3               41                    1770.0            52.0   
4               52                     777.0            60.0   

   delai_standard_prevu_min   ecart_min  taux_realisation_%  \
0                 38.375990    1.624010          104.231837   
1                108.168688   -3

In [27]:
# Déterminer automatiquement le chemin du Bureau
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")

# Créer un dossier de sortie sur le Bureau
output_folder = os.path.join(desktop_path, "outputs_modele_preparation")
os.makedirs(output_folder, exist_ok=True)

print("\n=== DOSSIER DE SORTIE ===")
print(output_folder)

# Export 1 : base détaillée
data.to_excel(os.path.join(output_folder, "01_commandes_avec_standard.xlsx"), index=False)
data.to_csv(os.path.join(output_folder, "01_commandes_avec_standard.csv"), index=False, encoding="utf-8-sig")

# Export 2 : standards par classes
standards_classes.to_excel(os.path.join(output_folder, "02_standards_par_classes.xlsx"), index=False)
standards_classes.to_csv(os.path.join(output_folder, "02_standards_par_classes.csv"), index=False, encoding="utf-8-sig")

# Export 3 : performance du modèle
performance_modele.to_excel(os.path.join(output_folder, "03_performance_modele.xlsx"), index=False)
performance_modele.to_csv(os.path.join(output_folder, "03_performance_modele.csv"), index=False, encoding="utf-8-sig")

# Export 4 : résumé pour Power BI
resume_bi.to_excel(os.path.join(output_folder, "04_resume_pour_bi.xlsx"), index=False)
resume_bi.to_csv(os.path.join(output_folder, "04_resume_pour_bi.csv"), index=False, encoding="utf-8-sig")

print("\n=== EXPORT TERMINÉ AVEC SUCCÈS ===")
print("Les fichiers ont été enregistrés sur votre Bureau dans le dossier :")
print(output_folder)


=== DOSSIER DE SORTIE ===
C:\Users\AdMin\Desktop\outputs_modele_preparation

=== EXPORT TERMINÉ AVEC SUCCÈS ===
Les fichiers ont été enregistrés sur votre Bureau dans le dossier :
C:\Users\AdMin\Desktop\outputs_modele_preparation
